In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

experiment = mlflow.set_experiment("/random-forest-hipervalorizacion")

with mlflow.start_run(experiment_id=experiment.experiment_id):
    
    print("--- 1. CARGA Y PREPARACIÓN DE DATOS ---")
    matriz = "/Volumes/workspace/default/datos_bogota/matriz_barrios_unificada.csv"
    df = pd.read_csv(matriz)
    
    # Separar datos supervisados vs inferencia
    df_supervised = df[df['v_ref_2026'].notnull()].copy()
    df_inference = df[df['v_ref_2026'].isnull()].copy()

    # Preparar Matriz X e y
    y_sup = df_supervised['v_ref_2026']
    cols_a_excluir = ['barrio', 'geometry', 'v_ref_2015', 'v_ref_2026', 'num_listings_airbnb', 'area_km2']
    X_sup = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])
    X_sup = pd.get_dummies(X_sup, columns=['localidad'], drop_first=True, dtype=int)

    # Train / Test Split
    X_train, X_test, y_train, y_test = train_test_split(X_sup, y_sup, test_size=0.2, random_state=42)

    # Definición de Hiperparámetros y registro en MLflow
    params = {
        "n_estimators": 500,
        "max_depth": 5,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "random_state": 42
    }
    mlflow.log_params(params)

    # Entrenamiento
    rf_model = RandomForestRegressor(**params, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    # Predicción y Evaluación
    y_pred_test = rf_model.predict(X_test)
    mse_rf = mean_squared_error(y_test, y_pred_test)
    rmse_rf = np.sqrt(mse_rf)
    mae_rf = mean_absolute_error(y_test, y_pred_test)
    r2_rf = r2_score(y_test, y_pred_test)

    # Registrar métricas en MLflow
    mlflow.log_metric("mse", mse_rf)
    mlflow.log_metric("rmse", rmse_rf)
    mlflow.log_metric("mae", mae_rf)
    mlflow.log_metric("r2", r2_rf)

    print(f"--- RESULTADOS RANDOM FOREST --- R2: {r2_rf:.4f} | MAE: {mae_rf:,.2f}")

    # Inferencia, consolidación y formato final
    X_inf = df_inference.drop(columns=[col for col in cols_a_excluir if col in df_inference.columns])
    X_inf = pd.get_dummies(X_inf, columns=['localidad'], drop_first=True, dtype=int)
    X_inf = X_inf.reindex(columns=X_train.columns, fill_value=0)
    df_inference['v_ref_2026_predicho'] = rf_model.predict(X_inf)

    X_sup_completo = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])
    X_sup_completo = pd.get_dummies(X_sup_completo, columns=['localidad'], drop_first=True, dtype=int)
    X_sup_completo = X_sup_completo.reindex(columns=X_train.columns, fill_value=0)
    df_supervised['v_ref_2026_predicho'] = rf_model.predict(X_sup_completo)

    df_final_maestro = pd.concat([df_supervised, df_inference], ignore_index=True)
    df_final_maestro['hipervalorizacion_pct'] = ((df_final_maestro['v_ref_2026_predicho'] - df_final_maestro['v_ref_2025']) / df_final_maestro['v_ref_2025']) * 100
    df_final_maestro['tipo_registro'] = df_final_maestro['v_ref_2026'].apply(lambda x: 'Histórico Oficial' if pd.notnull(x) else 'Imputado / Predicho (Huérfano)')
    
    df_final_maestro['v_ref_2026_predicho'] = df_final_maestro['v_ref_2026_predicho'].round(0)
    df_final_maestro['hipervalorizacion_pct'] = df_final_maestro['hipervalorizacion_pct'].round(2)

    # Registrar el modelo entrenado en MLflow
    mlflow.sklearn.log_model(rf_model, "random_forest_model")